# 第 11 章 · Human-in-the-loop：InteractiveAgent

**这一章你会得到什么**：理解 `InteractiveAgent` 如何在**不改主循环**的前提下，用继承 + 覆盖插入“人”的确认环节；看清 human / confirm / yolo 三种模式的分界。

## 📖 对照源码（在 IDE 里打开这些文件，边看边跑）

- `src/minisweagent/agents/interactive.py` **L24–34** — `InteractiveAgentConfig` + 三种模式
- `src/minisweagent/agents/interactive.py` **L162–182** — `_should_ask_confirmation` / `_ask_confirmation_or_interrupt`（确认逻辑）
- `src/minisweagent/agents/interactive.py` **L124–139** — `execute_actions`（try/finally 保留 observation）
- `src/minisweagent/agents/interactive.py` **L96–107** — `_stdin_is_interactive`（非交互环境自保护）

> 快捷：代码格里 `函数名??` 直接打印源码；或用 `show_source("相对路径", 起始行, 结束行)`。

In [ ]:
import os, sys
from pathlib import Path
os.environ["MSWEA_SILENT_STARTUP"] = "1"
REPO = Path(r"/Users/xinranzhao/Documents/llm-study/books/mini-swe-agent-source-guide/mini-swe-agent")
SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
os.chdir(REPO)
import minisweagent
print("mini-SWE-agent:", minisweagent.__version__)

In [ ]:
def show_source(rel_path: str, start: int, end: int) -> None:
    lines = (REPO / rel_path).read_text().splitlines()
    end = min(end, len(lines))
    w = len(str(end))
    for n in range(start, end + 1):
        print(f"{n:>{w}}  {lines[n - 1]}")

## 概念：三种模式

- **human**：命令由用户敲，模型不执行
- **confirm**（默认）：模型提命令，用户逐条确认
- **yolo**：模型命令直接执行，不确认

`InteractiveAgent` 继承 `DefaultAgent`，只覆盖 `query` / `step` / `execute_actions` 来插入交互，主循环 `run()` 一行没改。

In [ ]:
show_source("src/minisweagent/agents/interactive.py", 24, 34)

## 实验 1：确认逻辑是纯函数——不需要真的输入也能测

`_should_ask_confirmation(action)` 决定某条命令要不要弹确认。我们造一个带白名单的 agent 直接调它。

In [ ]:
from minisweagent.agents.interactive import InteractiveAgent
from minisweagent.models.test_models import DeterministicToolcallModel
from minisweagent.environments.local import LocalEnvironment

agent = InteractiveAgent(
    DeterministicToolcallModel(outputs=[]),
    LocalEnvironment(),
    system_template="s", instance_template="{{task}}",
    mode="confirm", whitelist_actions=["ls.*", "cat .*"],
)
print("confirm 模式 + 白名单命中 ls -la:", agent._should_ask_confirmation("ls -la"))
print("confirm 模式 + 未白名单 rm -rf /:", agent._should_ask_confirmation("rm -rf /"))

## 实验 2：切到 yolo 模式后，任何命令都不再确认

In [ ]:
agent.config.mode = "yolo"
print("yolo 模式 rm -rf /:", agent._should_ask_confirmation("rm -rf /"))
agent.config.mode = "human"
print("human 模式 rm -rf /:", agent._should_ask_confirmation("rm -rf /"))

## 观察点
- 白名单用**正则**匹配（`re.match`），所以 `"ls.*"` 会放行所有 `ls ...`。这是“安全但不烦人”的折中。
- 只有 `confirm` 模式且未命中白名单才弹确认；`yolo`/`human` 都直接返回 False。
- 用户拒绝命令时，`InteractiveAgent` 抛 `UserInterruption`（属于 `InterruptAgentFlow`），把拒绝理由作为消息喂回模型——复用了第 4 章那套异常控制流。

## 实验 3：非交互环境的自我保护

`_stdin_is_interactive()` 判断有没有真终端。在 notebook / CI / 沙箱里它返回 False，
于是 `--yolo` 跑到 LimitsExceeded 时会**干净退出**而不是卡在 `input()` 上崩溃。

In [ ]:
print("当前 stdin 是交互终端吗:", InteractiveAgent._stdin_is_interactive())

## 动手（思考题，不用写代码）

`InteractiveAgent` 覆盖了 `execute_actions` 并用 `try/finally` 包住执行。
回看第 4 章：为什么“确认被拒绝/提交”这类中断，也必须保证已产生的 observation 被追加进历史？
（提示：轨迹完整性 + 下一轮模型要看到“发生了什么”。）

In [ ]:
show_source("src/minisweagent/agents/interactive.py", 124, 139)

## 闭卷检查
1. 三种模式分别是什么？
2. `InteractiveAgent` 为了插入交互，覆盖了哪几个方法？主循环改了吗？
3. 为什么非交互环境要特判 `_stdin_is_interactive`？